# Technical Note P12: SciPy Interpolation, Integration, and Numerical Differentiation

**Wook Kang · Revised English edition · Version 1.0.0 · 3 October 2026**

This note connects a mathematical task to a numerical object, an implementation, and evidence about the result. It is an instructional computational document. Demonstration parameters are illustrative unless explicitly stated otherwise.

Run from a fresh kernel in cell order. See [the series README](../README.md) for setup and [editorial and verification notes](../docs/EDITORIAL_NOTES.md) for scope and limitations. The English prose has been reorganized and expanded from the supplied Korean notebook; this is an adapted edition rather than a line-by-line translation.

**Reading question:** What is given, what is unknown, what operation relates them, and what independent check can assess the output?

## Core mathematical structure

$$
\boxed{
\text{interpolation}
\quad
\text{integration}
\quad
\text{differentiation}
}
$$

In [1]:
# Reproducible display and scale-aware comparison.
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
def scaled_allclose(a, b, rtol=1e-12, atol=None, **kwargs):
    """Compare numerical examples using an absolute tolerance scaled to b."""
    a, b = np.asarray(a), np.asarray(b)
    scale = float(np.max(np.abs(b))) if b.size else 0.0
    if atol is None:
        atol = 1e-14 * max(scale, np.finfo(float).tiny)
    return np.allclose(a, b, rtol=rtol, atol=atol, **kwargs)


## 1. Three different tasks

Interpolation estimates values between samples, integration accumulates a quantity, and differentiation estimates a local rate of change. Their sensitivities to resolution and noise differ.

$$
(x_i,y_i)\rightarrow \tilde y(x)
$$

$$
I=\int_a^b f(x)\,dx
$$

$$
f'(x)
$$

$$
\boxed{\text{interpolation}\neq\text{fitting}}
$$

$$
\boxed{\text{integration tends to smooth}}
\qquad
\boxed{\text{differentiation tends to amplify noise}}
$$

In [2]:
import numpy as np
from scipy import interpolate, integrate

## 2. Linear interpolation

Join neighbouring values with straight segments. Use sorted unique coordinates and specify how queries outside the observed interval are handled.

$$
\tilde y(x)=
y_i+
\frac{x-x_i}{x_{i+1}-x_i}(y_{i+1}-y_i)
$$

In [3]:
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([0.0, 1.0, 0.0, 1.0])

def f_lin(xq):
    return np.interp(xq, x, y)
xq = np.linspace(0, 3, 13)
yq = f_lin(xq)

print("max nodal error =", np.max(np.abs(f_lin(x)-y)))
print("query values =", yq)

max nodal error = 0.0
query values = [0.   0.25 0.5  0.75 1.   0.75 0.5  0.25 0.   0.25 0.5  0.75 1.  ]


## 3. Extrapolation

Extrapolation uses a model beyond the data support. The default behaviour varies by interface and should be stated explicitly.

$$
\boxed{
\text{interpolation}=
\text{inside known domain}
}
$$

$$
\boxed{
\text{extrapolation}=
\text{outside known domain}
}
$$

In [4]:
f_ext = interpolate.make_interp_spline(x, y, k=1)  # Explicit linear extrapolation demonstration.
print("f(-1) =", f_ext(-1.0))
print("f(4) =", f_ext(4.0))

f(-1) = -1.0
f(4) = 2.0


## 4. Cubic splines

CubicSpline supplies a piecewise cubic function with continuous first and second derivatives. Endpoint conditions influence the interpolant and its derivatives.

In [5]:
x = np.linspace(0, 2*np.pi, 9)
y = np.sin(x)

cs = interpolate.CubicSpline(x, y)

xq = np.linspace(0, 2*np.pi, 200)
dcs = cs.derivative()

print("max nodal error =", np.max(np.abs(cs(x)-y)))
print("max derivative error =", np.max(np.abs(dcs(xq)-np.cos(xq))))
print("integral 0..pi =", cs.integrate(0, np.pi))

max nodal error = 4.87082071118958e-18
max derivative error = 0.0592242425628422
integral 0..pi = 2.001822131885034


## 5. PCHIP

PCHIP preserves monotonicity within monotone data segments and avoids many cubic overshoots. It does not guarantee physical accuracy or remove noisy observations.

$$
\boxed{
\text{smoothness}
\neq
\text{physical shape preservation}
}
$$

In [6]:
x = np.array([0,1,2,3,4], dtype=float)
y = np.array([0,1,1.2,1.25,1.3], dtype=float)

pchip = interpolate.PchipInterpolator(x, y)
akima = interpolate.Akima1DInterpolator(x, y)

print("PCHIP at 2.5 =", pchip(2.5))
print("Akima at 2.5 =", akima(2.5))

PCHIP at 2.5 = 1.22875
Akima at 2.5 = 1.225


## 6. Interpolation and fitting

Interpolation passes through sample values; fitting trades exact passage for a model of the trend or noise. Exact interpolation of noisy measurements can amplify derivative errors.

$$
\tilde y(x_i)=y_i
$$

$$
\tilde y(x_i)\approx y_i
$$

## 7. Smoothing splines

Smoothing introduces an additional modeling choice. The revised examples use make_splrep rather than the legacy UnivariateSpline interface; smoothing strength is not a universal physical parameter.

$$
\boxed{
\text{smoothing}
\neq
\text{free noise removal}
}
$$

In [7]:
rng = np.random.default_rng(2)
x = np.linspace(0, 2*np.pi, 30)
y_noisy = np.sin(x) + 0.12*rng.normal(size=x.size)

spl_interp = interpolate.make_splrep(x, y_noisy, s=0)
spl_smooth = interpolate.make_splrep(x, y_noisy, s=0.4)

print("interpolating spline max nodal residual =",
      np.max(np.abs(spl_interp(x)-y_noisy)))
print("smoothing spline RSS =",
      np.sum((spl_smooth(x)-y_noisy)**2))

interpolating spline max nodal residual = 2.220446049250313e-16
smoothing spline RSS = 0.3999656163938344


## 8. Regular-grid interpolation

RegularGridInterpolator needs coordinates ordered consistently with the field axes. An x-first field requires a corresponding x-first coordinate tuple.

$$
u=u(x,y)
$$

```python
RegularGridInterpolator((x,y), U)
```

```python
U.shape == (len(x), len(y))
```

In [8]:
x = np.linspace(0, 1, 11)
y = np.linspace(0, 2, 21)
X, Y = np.meshgrid(x, y, indexing="ij")
U = np.sin(np.pi*X)*np.cos(np.pi*Y/2)

interp2 = interpolate.RegularGridInterpolator((x, y), U)

points = np.array([[0.25,0.5], [0.70,1.2]])
print(interp2(points))

[ 0.49384417 -0.25      ]


## 9. Adaptive quadrature

Quad evaluates a callable at chosen points and estimates integration error. Singularities, discontinuities, and oscillations require appropriate information or a different approach.

$$
I=\int_a^b f(x)\,dx
$$

In [9]:
f = lambda x: np.exp(-x**2)

I, err_est = integrate.quad(f, 0, 1)

print("integral =", I)
print("estimated abs error =", err_est)

I_inf, err_inf = integrate.quad(lambda x: np.exp(-x), 0, np.inf)
print("0..inf exp(-x) =", I_inf)

integral = 0.7468241328124271
estimated abs error = 8.291413475940725e-15
0..inf exp(-x) = 1.0000000000000002


## 10. Sampled-data integration

Trapezoid and Simpson integrate supplied values. Their accuracy cannot recover unresolved variations between samples.

$$
(x_i,y_i)
$$

In [10]:
x = np.linspace(0, 1, 11)
y = x**2

I_trap = integrate.trapezoid(y, x)
I_simp = integrate.simpson(y, x=x)

print("trapezoid =", I_trap)
print("Simpson   =", I_simp)
print("exact     =", 1/3)

trapezoid = 0.33499999999999996
Simpson   = 0.33333333333333337
exact     = 0.3333333333333333


## 11. Cumulative integration

Cumulative_trapezoid builds an accumulated history. Include the initial value convention and verify the units of the integrated quantity.

$$
I(x)=\int_{x_0}^{x}f(s)\,ds
$$

In [11]:
t = np.linspace(0, 10, 101)
q = np.exp(-0.3*t)

Q = integrate.cumulative_trapezoid(q, t, initial=0.0)

print("final accumulated quantity =", Q[-1])

final accumulated quantity = 3.1676139884434713


## 12. Finite-difference derivatives

Forward, backward, and centred formulas approximate derivatives under smoothness and spacing assumptions. Their boundary stencils differ from their interior formulas.

$$
f'(x_i)\approx\frac{f_{i+1}-f_i}{h}
$$

$$
f'(x_i)\approx\frac{f_i-f_{i-1}}{h}
$$

$$
f'(x_i)\approx\frac{f_{i+1}-f_{i-1}}{2h}
$$

In [12]:
x = np.linspace(0, 1, 11)
f = x**2
h = x[1]-x[0]

df_forward = (f[1:] - f[:-1])/h
df_central = (f[2:] - f[:-2])/(2*h)

print("forward =", df_forward[:4])
print("central =", df_central[:4])

forward = [0.1 0.3 0.5 0.7]
central = [0.2 0.4 0.6 0.8]


## 13. Centred-difference convergence

A smooth function on a uniform grid exhibits second-order truncation error for the centred first derivative. Floating-point cancellation eventually limits improvement as the step decreases.

$$
O(h^2)
$$

In [13]:
for N in [21, 41, 81, 161]:
    x = np.linspace(0, 1, N)
    h = x[1]-x[0]
    f = np.sin(x)

    df_num = (f[2:] - f[:-2])/(2*h)
    df_exact = np.cos(x[1:-1])

    err = np.max(np.abs(df_num-df_exact))
    print(f"N={N:3d}, h={h:.3e}, max error={err:.3e}")

N= 21, h=5.000e-02, max error=4.161e-04
N= 41, h=2.500e-02, max error=1.041e-04
N= 81, h=1.250e-02, max error=2.604e-05
N=161, h=6.250e-03, max error=6.510e-06


## 14. Gradient of sampled data

Np.gradient accepts coordinate arrays and can handle nonuniform spacing. It estimates sample derivatives rather than imposing a conservative flux discretization.

In [14]:
x = np.linspace(0, 1, 51)**2
f = np.sin(x)

df = np.gradient(f, x, edge_order=2)

print("max error =", np.max(np.abs(df-np.cos(x))))

max error = 0.000292264123439967


## 15. Noise amplification

Differences divide measurement perturbations by a step length. Smaller spacing can increase derivative noise even while reducing truncation error.

$$
\frac{\epsilon_{i+1}-\epsilon_i}{h}
$$

$$
\boxed{
h\downarrow
\not\Rightarrow
\text{derivative accuracy always improves}
}
$$

In [15]:
rng = np.random.default_rng(0)

x = np.linspace(0, 2*np.pi, 200)
f_true = np.sin(x)
f_noisy = f_true + 0.01*rng.normal(size=x.size)

df_true = np.cos(x)
df_noisy = np.gradient(f_noisy, x)

print("signal noise std =", np.std(f_noisy-f_true))
print("derivative error std =", np.std(df_noisy-df_true))

signal noise std = 0.009611788685151808
derivative error std = 0.20686906268795185


## 16. Smooth before differentiating

A fitted smooth function provides a derivative estimate with additional assumptions. Check sensitivity to smoothing strength and to boundary behaviour.

In [16]:
rng = np.random.default_rng(3)

x = np.linspace(0, 2*np.pi, 60)
y = np.sin(x) + 0.08*rng.normal(size=x.size)

spl = interpolate.make_splrep(x, y, s=0.35)
dspl = spl.derivative()

xq = np.linspace(0, 2*np.pi, 300)
err = np.sqrt(np.mean((dspl(xq)-np.cos(xq))**2))

print("RMS derivative error after smoothing =", err)

RMS derivative error after smoothing = 0.21782850994085734


## 17. Savitzky-Golay derivatives

A local polynomial window provides smoothing and differentiation together. Window length, polynomial degree, sample spacing, and endpoint treatment influence the result.

In [17]:
from scipy.signal import savgol_filter

x = np.linspace(0, 2*np.pi, 101)
rng = np.random.default_rng(4)
y = np.sin(x) + 0.05*rng.normal(size=x.size)

dx = x[1]-x[0]
dy_sg = savgol_filter(
    y,
    window_length=15,
    polyorder=3,
    deriv=1,
    delta=dx,
)

print("RMS derivative error =",
      np.sqrt(np.mean((dy_sg-np.cos(x))**2)))

RMS derivative error = 0.15839500402265042


## 18. Data differences and interpolant derivatives

A finite difference differentiates samples through a stencil; a spline derivative differentiates the selected interpolation model. They answer related but different numerical questions.

$$
\text{samples}
\rightarrow
\text{local difference}
$$

$$
\text{samples}
\rightarrow
\text{continuous approximation}
\rightarrow
\text{derivative}
$$

$$
\boxed{
\text{differentiate data}
\neq
\text{differentiate fitted/interpolated function}
}
$$

## 19. Finite-difference sensitivities

Perturb a scaled parameter and compare forward responses. The step must be large enough to exceed solver noise and small enough for local linearization.

$$
S_j=
\frac{\partial y}{\partial\theta_j}
$$

$$
S_j
\approx
\frac{
y(\theta_j+h)-y(\theta_j-h)
}{2h}
$$

In [18]:
t = np.linspace(0, 3, 10)

def model(theta):
    return np.exp(-theta*t)

theta = 0.7
h = 1e-5

S_fd = (model(theta+h)-model(theta-h))/(2*h)
S_exact = -t*np.exp(-theta*t)

print("max sensitivity error =", np.max(np.abs(S_fd-S_exact)))

max sensitivity error = 5.5315918512377493e-11


## 20. Callable and sampled integration

Callable quadrature can request additional values. Sampled-data integration is limited by the observations already available; quadrature error estimates do not include measurement uncertainty.

$$
f(x)\text{ callable}
\rightarrow
\text{adaptive evaluation}
$$

$$
(x_i,y_i)
\rightarrow
\text{integration of supplied samples}
$$

$$
\boxed{
\text{function quadrature}
\neq
\text{sampled-data integration}
}
$$

## 21. Integrate a two-dimensional field

Integrate along one coordinate axis and then the other with matching coordinates. Confirm whether stored values are nodal samples or cell averages.

$$
U(x,y)
$$

In [19]:
x = np.linspace(0, 1, 101)
y = np.linspace(0, 2, 121)

X, Y = np.meshgrid(x, y, indexing="ij")
U = X + Y

Iy = integrate.trapezoid(U, y, axis=1)
Ixy = integrate.trapezoid(Iy, x)

print("2D integral =", Ixy)
print("exact =", 3.0)

2D integral = 2.999999999999999
exact = 3.0


## 22. Axis-aware differentiation

The derivative direction follows the declared storage axis. Passing the wrong spacing can produce a plausible array with incorrect values and units.

```python
U.shape == (Nx,Ny)
```

```python
np.gradient(U, x, axis=0)
```

```python
np.gradient(U, y, axis=1)
```

In [20]:
dUdx = np.gradient(U, x, axis=0, edge_order=2)
dUdy = np.gradient(U, y, axis=1, edge_order=2)

print("mean dU/dx =", np.mean(dUdx))
print("mean dU/dy =", np.mean(dUdy))

mean dU/dx = 1.0
mean dU/dy = 0.9999999999999997


## 23. Boundary tables as callables

An interpolated ambient history can be called by a time integrator. Specify units, missing values, discontinuities, and extrapolation policy.

$$
T_\infty(t)
$$

In [21]:
t_table = np.array([0,1,3,6,10], dtype=float)
T_table = np.array([20,40,60,50,30], dtype=float)

T_inf = interpolate.PchipInterpolator(t_table, T_table)

print("T_inf(2.0) =", T_inf(2.0))

T_inf(2.0) = 53.46153846153846


## 24. Material tables as constitutive laws

Interpolation can turn measured property samples into a callable. Check positivity, monotonicity where justified, state dependence, and uncertainty rather than trusting smoothness alone.

$$
D=D(M)
$$

In [22]:
M_data = np.array([0.05,0.10,0.15,0.20,0.30])
D_data = np.array([0.8,1.0,1.3,1.8,3.0])*1e-10

D_of_M = interpolate.PchipInterpolator(M_data, D_data)

M_query = np.linspace(0.05,0.30,6)
print(D_of_M(M_query))

[8.00000000e-11 1.00000000e-10 1.30000000e-10 1.80000000e-10
 2.36833333e-10 3.00000000e-10]


## 25. Interpolation failure modes

Unsorted or duplicate coordinates, accidental extrapolation, overshoot, and exact fitting of noise can undermine an interpolant. Inspect both the values and intended physical domain.

$$
\boxed{
\text{smooth numerical curve}
\neq
\text{certain physical law}
}
$$

## 26. Integration failure modes

Wrong units, unresolved oscillations, discontinuities, and inadequate sampling can dominate the error. A small reported numerical estimate is not a complete uncertainty statement.

## 27. Differentiation failure modes

Noise, irregular spacing, boundaries, nonsmooth signals, and a poor perturbation size can dominate derivative error. Plot and assess these effects separately.

## 28. One bridge between data and calculus

Interpolation creates a model between samples, integration aggregates it, and differentiation interrogates its local shape. Each operation requires a stated interpretation.

$$
\text{discrete samples}
\rightarrow
\text{continuous approximation}
$$

$$
\text{local values}
\rightarrow
\text{accumulated quantity}
$$

$$
\text{neighboring values}
\rightarrow
\text{local rate of change}
$$

## 29. Operation map

Use np.interp for simple linear interpolation, CubicSpline or PCHIP for suitable curves, make_splrep for smoothing, and the integration or derivative tools appropriate to the data representation.

| Problem | SciPy / NumPy |  
| --- | --- |  
| linear interpolation | `np.interp` (linear 1D) |  
| cubic spline | `CubicSpline` |  
| monotone interpolation | `PchipInterpolator` |  
| local smooth interpolation | `Akima1DInterpolator` |  
| smoothing spline | `make_splrep` |  
| regular-grid interpolation | `RegularGridInterpolator` |  
| adaptive quadrature | `integrate.quad` |  
| trapezoidal integration | `integrate.trapezoid` |  
| Simpson integration | `integrate.simpson` |  
| cumulative integration | `cumulative_trapezoid` |  
| sampled derivative | `np.gradient` |  
| spline derivative | `spline.derivative()` |  
| smoothed derivative | smoothing spline / Savitzky-Golay |

## 30. Ideas to retain

Distinguish interpolation from extrapolation and fitting; distinguish numerical estimates from data uncertainty; assess smoothing and derivative step choices explicitly.

$$
\boxed{
\text{interpolation}
\neq
\text{fitting}
}
$$

$$
\boxed{
h\downarrow
\not\Rightarrow
\text{derivative accuracy always improves}
}
$$

$$
\boxed{
\text{smooth numerical curve}
\neq
\text{certain physical law}
}
$$

## 31. Continue to P13

The final note combines numerical checks, plots, data tables, environment records, and a clear statement of what has been verified.

$$
\boxed{
\text{compute}
\rightarrow
\text{inspect}
\rightarrow
\text{verify}
\rightarrow
\text{record}
}
$$

## Further work

1. Change one parameter or discretization choice and predict the effect before running.
2. Write the input and output shapes, units, and field locations.
3. Construct a deliberately invalid input and make the calculation report it clearly.
4. Separate an execution check from an accuracy check and from any physical claim.

## Primary documentation

- [SciPy interpolation](https://docs.scipy.org/doc/scipy/reference/interpolate.html)
- [NumPy user guide](https://numpy.org/doc/stable/user/)

These are living documentation links. The accompanying requirements and verification report identify the versions actually executed for this edition.

## AI assistance and responsibility

The English adaptation, editorial supplementation, and code review were prepared with AI assistance. Wook Kang retains responsibility for reviewing the explanations, examples, and any subsequent research application. No new experimental validation is claimed.